# Spaceship Titanic — CatBoost workflow

A high-quality tabular baseline with consistent feature engineering, out-of-fold validation, and group-aware prediction calibration.

## Strategy

CatBoost natively learns from categorical features and missing numerical values. We parse passenger groups, cabins, families, and spending behaviour, then validate both the raw model and an optional group-consensus rule using out-of-fold predictions. The final choice is based on validation accuracy—not assumptions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 42
N_SPLITS = 5
DATA_DIR = Path('../data')
SPEND_COLUMNS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
sns.set_theme(style='whitegrid')

## Load data

In [2]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv')
test_raw = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')
display(train_raw.head())
display(train_raw.isna().sum().sort_values(ascending=False).to_frame('missing'))

Train: (8693, 14) | Test: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


,missing
CryoSleep,217
ShoppingMall,208
VIP,203
HomePlanet,201
Name,200
Cabin,199
VRDeck,188
Spa,183
FoodCourt,183
Destination,182


## Feature engineering

All features are derived without the target. Missing booking-level fields are filled from other passengers in the same booking; this is a legitimate test-time signal because those passenger records are available when making predictions.

In [3]:
def group_mode(values):
    values = values.dropna()
    return values.mode().iat[0] if not values.mode().empty else np.nan

def build_features(frame, reference):
    df = frame.copy()
    ref_group = reference['PassengerId'].str.split('_').str[0]
    parts = df['PassengerId'].str.split('_', expand=True)
    df['GroupId'] = parts[0]
    df['GroupPosition'] = pd.to_numeric(parts[1], errors='coerce')
    df['GroupSize'] = df['GroupId'].map(ref_group.value_counts())
    df['IsGroupLeader'] = (df['GroupPosition'] == 1).astype('int8')

    # Recover sparse booking data from known members of the same group.
    for column in ['HomePlanet', 'Cabin', 'Destination', 'CryoSleep', 'VIP']:
        values = pd.DataFrame({'group': ref_group, 'value': reference[column]}).groupby('group')['value'].agg(group_mode)
        df[column] = df[column].fillna(df['GroupId'].map(values))

    cabin = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = cabin[0]
    df['CabinNumber'] = pd.to_numeric(cabin[1], errors='coerce')
    df['Side'] = cabin[2]
    df['CabinNumberLog'] = np.log1p(df['CabinNumber'])
    df['CabinZone'] = pd.cut(df['CabinNumber'], [-1, 300, 700, 1200, 2000, np.inf], labels=False)

    df['Surname'] = df['Name'].str.split().str[-1]
    ref_surname = reference['Name'].str.split().str[-1]
    df['SurnameSize'] = df['Surname'].map(ref_surname.value_counts())

    spend = df[SPEND_COLUMNS].fillna(0)
    df['TotalSpend'] = spend.sum(axis=1)
    df['LuxurySpend'] = spend[['FoodCourt', 'Spa', 'VRDeck']].sum(axis=1)
    df['ServiceSpend'] = spend[['RoomService', 'ShoppingMall']].sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype('int8')
    df['CryoSleep'] = df['CryoSleep'].fillna(df['NoSpend'].map({1: True, 0: False}))
    for column in SPEND_COLUMNS + ['TotalSpend', 'LuxurySpend', 'ServiceSpend']:
        df[f'{column}Log'] = np.log1p(df[column].fillna(0))
    df['AgeBand'] = pd.cut(df['Age'], [-1, 12, 18, 30, 45, 60, np.inf], labels=['child','teen','young_adult','adult','middle_age','senior'])

    return df.drop(columns=['PassengerId', 'Cabin', 'Name', 'GroupId', 'Surname'], errors='ignore')

reference = pd.concat([train_raw.drop(columns='Transported'), test_raw], ignore_index=True)
X = build_features(train_raw.drop(columns='Transported'), reference)
X_test = build_features(test_raw, reference)
y = train_raw['Transported'].astype(int)

categorical_columns = X.select_dtypes(exclude='number').columns.tolist()
for column in categorical_columns:
    X[column] = X[column].astype('object').fillna('Missing').astype(str)
    X_test[column] = X_test[column].astype('object').fillna('Missing').astype(str)

assert X.columns.equals(X_test.columns)
print(f'Features: {X.shape[1]} ({len(categorical_columns)} categorical)')
display(X.head())

C:\Users\Reza.M\AppData\Local\Temp\ipykernel_24996\501376136.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep'] = df['CryoSleep'].fillna(df['NoSpend'].map({1: True, 0: False}))


Features: 32 (7 categorical)


C:\Users\Reza.M\AppData\Local\Temp\ipykernel_24996\501376136.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['CryoSleep'] = df['CryoSleep'].fillna(df['NoSpend'].map({1: True, 0: False}))
C:\Users\Reza.M\AppData\Local\Temp\ipykernel_24996\501376136.py:49: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[column] = X[column].astype('object').fillna('Missing').astype(str)
C:\Users\Reza.M\AppData\Local\Temp\ipykernel_24996\501376136.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a futur

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,...,NoSpend,RoomServiceLog,FoodCourtLog,ShoppingMallLog,SpaLog,VRDeckLog,TotalSpendLog,LuxurySpendLog,ServiceSpendLog,AgeBand
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,...,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,adult
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,...,0,4.700480,2.302585,3.258097,6.309918,3.806662,6.602588,6.401917,4.905275,young_adult
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,...,0,3.784190,8.182280,0.000000,8.812248,3.912023,9.248021,9.243872,3.784190,middle_age
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,...,0,0.000000,7.157735,5.918894,8.110728,5.267858,8.551981,8.477620,5.918894,adult
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,...,0,5.717028,4.262680,5.023881,6.338594,1.098612,6.995766,6.458338,6.120297,teen


## Cross-validation and model selection

The threshold and group-consensus rule are selected from out-of-fold predictions. This makes the decision auditable and prevents optimizing directly on Kaggle test predictions.

In [4]:
def make_model(seed):
    return CatBoostClassifier(
        iterations=1800, learning_rate=0.03, depth=7, l2_leaf_reg=5,
        loss_function='Logloss', eval_metric='Accuracy', random_seed=seed,
        verbose=False, allow_writing_files=False, thread_count=-1,
    )

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_probability = np.zeros(len(X))
best_iterations = []
fold_rows = []

for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = make_model(RANDOM_STATE + fold)
    model.fit(
        X.iloc[fit_idx], y.iloc[fit_idx], cat_features=categorical_columns,
        eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]), early_stopping_rounds=150,
    )
    oof_probability[valid_idx] = model.predict_proba(X.iloc[valid_idx])[:, 1]
    best_iterations.append(model.get_best_iteration())
    fold_rows.append({'fold': fold, 'best_iteration': model.get_best_iteration(),
                      'accuracy': accuracy_score(y.iloc[valid_idx], oof_probability[valid_idx] >= 0.5)})

cv_results = pd.DataFrame(fold_rows)
display(cv_results.style.format({'accuracy': '{:.4f}'}))
print(f"Raw OOF accuracy: {accuracy_score(y, oof_probability >= 0.5):.4f}")
print(f'Final iterations: {int(np.median(best_iterations))}')

,fold,best_iteration,accuracy
0,1,185,0.8160
1,2,281,0.8085
2,3,150,0.8200
3,4,573,0.8245
4,5,373,0.8096


Raw OOF accuracy: 0.8157
Final iterations: 281


In [5]:
def consensus_prediction(probability, passenger_ids, threshold):
    """Use a booking's mean probability only when the booking has multiple passengers."""
    groups = passenger_ids.str.split('_').str[0]
    group_mean = pd.Series(probability).groupby(groups.to_numpy()).transform('mean').to_numpy()
    group_size = groups.map(groups.value_counts()).to_numpy()
    adjusted = probability.copy()
    adjusted[group_size > 1] = group_mean[group_size > 1]
    return adjusted >= threshold

thresholds = np.arange(0.42, 0.59, 0.01)
selection = []
for threshold in thresholds:
    selection.append({
        'threshold': threshold,
        'raw_accuracy': accuracy_score(y, oof_probability >= threshold),
        'consensus_accuracy': accuracy_score(y, consensus_prediction(oof_probability, train_raw['PassengerId'], threshold)),
    })
selection = pd.DataFrame(selection)
display(selection.sort_values(['consensus_accuracy', 'raw_accuracy'], ascending=False).head(10))

best = selection.loc[selection[['raw_accuracy', 'consensus_accuracy']].max(axis=1).idxmax()]
use_consensus = best['consensus_accuracy'] > best['raw_accuracy']
best_threshold = float(best['threshold'])
print(f"Selected: {'group consensus' if use_consensus else 'raw probabilities'} at threshold {best_threshold:.2f}")

,threshold,raw_accuracy,consensus_accuracy
8,0.50,0.815714,0.756471
6,0.48,0.812033,0.756356
9,0.51,0.811457,0.755550
7,0.49,0.812723,0.755320
5,0.47,0.810307,0.754975
3,0.45,0.808812,0.754860
4,0.46,0.809502,0.754745
10,0.52,0.809617,0.752905
11,0.53,0.809962,0.752444
2,0.44,0.806971,0.752444


Selected: raw probabilities at threshold 0.50


## Fit on all labelled data and create submission

In [6]:
final_model = make_model(RANDOM_STATE)
final_model.set_params(iterations=int(np.median(best_iterations)))
final_model.fit(X, y, cat_features=categorical_columns)
test_probability = final_model.predict_proba(X_test)[:, 1]

if use_consensus:
    test_prediction = consensus_prediction(test_probability, test_raw['PassengerId'], best_threshold)
else:
    test_prediction = test_probability >= best_threshold

submission = pd.DataFrame({'PassengerId': test_raw['PassengerId'], 'Transported': test_prediction.astype(bool)})
assert submission['PassengerId'].equals(sample_submission['PassengerId'])
submission.to_csv(DATA_DIR / 'submission.csv', index=False)
display(submission.head())
print(f"Saved: {(DATA_DIR / 'submission.csv').resolve()}")

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


Saved: D:\Kaggle_Project\spaceship-titanic-ml\data\submission.csv


## Notes

This workflow should improve substantially on the 80% Extra Trees baseline, but no notebook can guarantee a particular Kaggle leaderboard score because the hidden test labels are unavailable. Use the OOF score to decide whether the observed improvement is real before submitting.